# Part c: Activation functions, depth and width

We compare the sigmoid, the ReLU and the leaky ReLU in the hidden layers, for networks of different depth and width.
Data, scaling, optimizer and number of epochs are the same as in part b, so only the activation and the architecture change.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # add Code/ so `src` can be imported

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import runge, make_runge_split, scale_data
from src.functions import *
from src.plotting import line_plot, heatmap, save_fig
from src.network import NeuralNetwork
from src.optimizers import Adam

## Data (same as part b)

In [ ]:
X_train, X_test, y_train, y_test = make_runge_split(n=100, sigma=0.1, test_size=0.2, seed=2026)
X_train_s, X_test_s, x_mean, x_std = scale_data(X_train, X_test)

x_plot = np.linspace(-1, 1, 300).reshape(-1, 1)
y_true = runge(x_plot)

## Activations and architectures


In [ ]:
# Activation functions and their derivatives
activations = { 
    "sigmoid":    (sigmoid, sigmoid_der),
    "ReLU":       (ReLU, ReLU_der),
    "leaky ReLU": (leaky_ReLU, leaky_ReLU_der)}

# hidden layer
architectures = [[10], [50], [100], [50, 50], [100, 100], [50] * 3, [50] * 5, [50] * 10]
arch_labels = ['10', '50', '100', '50-50', '100-100', '3x50', '5x50', '10x50']

## Train all combinations

For every activation function and architechture we train a network and store the training and test MSE.
All networks are trained with Adam ($\gamma = 0.01$), 500 epochs and batch size 14).

In [ ]:
train_mse = np.zeros((len(activations), len(architectures)))
test_mse  = np.zeros((len(activations), len(architectures)))
results = {}

for i, (act_name, (act, act_der)) in enumerate(activations.items()):
    for j, hidden in enumerate(architectures):
        nodes = len(hidden)
        net = NeuralNetwork(
            network_input_size=1,
            layer_output_sizes=hidden + [1],
            activation_funcs=[act] * nodes + [identity],
            activation_ders=[act_der] * nodes + [identity_der],
            cost_fun=mse,
            cost_der=mse_der,
            optimizer=Adam(gamma=0.01),     # new optimizer for every network
            seed=2026)
        history = net.train(X_train_s, y_train, epochs=500, batch_size=14,
                            rng=np.random.default_rng(2026))
        train_mse[i, j] = mse(net.predict(X_train_s), y_train)
        test_mse[i, j]  = mse(net.predict(X_test_s), y_test)
        results[(act_name, arch_labels[j])] = {
            "history": history,
            "plot": net.predict((x_plot - x_mean) / x_std)}
    

## Test and training MSE

Test and training MSE for every combination of activation function and architechture. If the training MSE is much lower than test MSE, the network is overfitting. The noise floor is $\sigma^2 = 0.01$.

In [ ]:
fig, ax = heatmap(test_mse, range(len(architectures)), range(len(activations)),
                  'hidden layers (nodes per layer)', 'activation', label="Test MSE")
ax.set_xticklabels(arch_labels, rotation=45)
ax.set_yticklabels(list(activations), rotation=0)
fig.set_size_inches(8, 4)
save_fig("c_heatmap_test")

In [ ]:
fig, ax = heatmap(train_mse, range(len(architectures)), range(len(activations)),
                  'hidden layers (nodes per layer)', 'activation', label="Train MSE")
ax.set_xticklabels(arch_labels, rotation=45)
ax.set_yticklabels(list(activations), rotation=0)
fig.set_size_inches(8, 4)
save_fig("c_heatmap_train")

## The deepest networks: fits and learning curves

Fitted curves and learning curves for the deepest networks, with 10 hidden layers with 50 nodes each.

In [ ]:
fig, ax = line_plot(
    x_plot,
    {
        "True Runge function": y_true,
        "NN 10x50, sigmoid":    results[("sigmoid", "10x50")]["plot"],
        "NN 10x50, ReLU":       results[("ReLU", "10x50")]["plot"],
        "NN 10x50, leaky ReLU": results[("leaky ReLU", "10x50")]["plot"]
    },
    xlabel="$x$", 
    ylabel="$y$", 
    data=(X_train, y_train), 
    name="c_fit_10x50")

epochs = np.arange(1, 501)
fig, ax = line_plot(
    epochs,
    {
        "sigmoid":    results[("sigmoid", "10x50")]["history"],
        "ReLU":       results[("ReLU", "10x50")]["history"],
        "leaky ReLU": results[("leaky ReLU", "10x50")]["history"]
    },
    xlabel="Epoch", 
    ylabel="Training MSE", 
    logy=True, 
    hlines={"noise floor": 0.01},
    name="c_learning_curve_10x50")


## Vanishing gradients

The size of $\partial C/\partial W$ in each layer of an untrained network with 10 hidden layers of 50 nodes. Backpropagation multiplies by the derivative of the activation function once per layer. For the sigmoid, $\sigma'(z) \le 1/4$, so the gradient shrinks on its way back to the first layers (vanishing gradients, Section 8.9 of the lecture notes).


In [ ]:
grad_norms = {}
for act_name, (act, act_der) in activations.items():
    net = NeuralNetwork(
        network_input_size=1,
        layer_output_sizes=[50] * 10 + [1],
        activation_funcs=[act] * 10 + [identity],
        activation_ders=[act_der] * 10 + [identity_der],
        cost_fun=mse,
        cost_der=mse_der,
        optimizer=Adam(gamma=0.01),
        seed=2026)
    grads = net.compute_gradient(X_train_s, y_train)
    grad_norms[act_name] = [np.linalg.norm(dW) for dW, db in grads]

# Kan velge å enten utskriften (fra week42monday.ipynb) eller plottet, trenger ikke begge.
    print(act_name)
    for l, g in enumerate(grad_norms[act_name], start=1):
        print(f"  layer {l:2d}:  |dC/dW| = {g:.1e}")
# ReLU ligger rett bak leaky ReLU 
fig, ax = line_plot(np.arange(1, 12), grad_norms, xlabel="layer", ylabel=r"$|\partial C/\partial W|$",
                    logy=True, name="c_gradient_norms")
